In [5]:
%%writefile app.py

import streamlit as st #steamlit creates we app interface
import pickle #we need to load the pkl file in app.py
import pandas as pd

with open("hybrid_recommender.pkl", "rb") as f: #Opens the saved recommender file.
    data = pickle.load(f) #Loads the saved dictionary from the hybrid_recommender.pkl file.

course_lookup = data["course_lookup"]
course_ids = data["course_ids"]
course_indices = data["course_indices"]
cosine_sim = data["cosine_sim"]
course_similarity = data["course_similarity"]
#Extracts each saved object.

def hybrid_recommend(course_id, top_n=5, content_weight=0.5): #Creates a function for recommendation
    cf_weight = 1 - content_weight #Calculates collaborative filtering weight,If content weight is 0.7, CF weight becomes 0.3.
    content_idx = course_indices[course_id] #Finds the row position of the selected course

    content_scores = list(enumerate(cosine_sim[content_idx])) #Gets content-based similarity scores for all courses.
                                                              #It compares the selected course with every other course using course features.
    cf_scores = list(enumerate(course_similarity[content_idx])) #Gets collaborative filtering similarity scores.

    hybrid_scores = [] #Creating empty list to store final combined scores

    for i in range(len(course_ids)): #loops through all courses
        if i != content_idx: #skips itself
            score = content_weight * content_scores[i][1] + cf_weight * cf_scores[i][1] #calculates final hybrid score
            hybrid_scores.append((i, score))#Stores course position and score

    hybrid_scores = sorted(hybrid_scores, key=lambda x: x[1], reverse=True) #Sorts courses from highest score to lowest

    top_indices = [x[0] for x in hybrid_scores[:top_n]] #Selects only the top_n course positions as given by user
    recommended_course_ids = [course_ids[i] for i in top_indices]#converts row position back to course IDs, to fetch course names

    recommendations = course_lookup[
        course_lookup["course_id"].isin(recommended_course_ids)
    ].copy() #Gets course details from course_lookup

    score_map = {course_ids[x[0]]: x[1] for x in hybrid_scores[:top_n]}
    recommendations["hybrid_score"] = recommendations["course_id"].map(score_map)
    #Creates a mapping between course ID and hybrid score
    #This lets us attach the score to each recommended course
    #Adds hybrid score as a new column

    return recommendations.sort_values("hybrid_score", ascending=False) #Returns final recommendations sorted by best score

st.title("Online Course Recommendation System") #Title of the app

selected_course = st.selectbox(
    "Select Course ID",
    course_lookup["course_id"].tolist()
) #create dropdown of course IDs

top_n = st.slider("Number of recommendations", 1, 10, 5) #Create slider to choose how many recommendations they want.

content_weight = st.slider("Content-Based Weight", 0.0, 1.0, 0.5) #Creates weight slider
#User can control balance between content-based and collaborative filtering.

if st.button("Recommend Courses"): #app runs when user clicks this button
    results = hybrid_recommend(selected_course, top_n, content_weight) #calls the hybrid recommender
    st.subheader("Recommended Courses") #3adds heading above the result
    st.dataframe(results) #Displays recommendations as a table

Overwriting app.py
